# Iterators, Generators & Lazy Pipelines: Beginner Guide

### 🌟 What Are Iterators & Generators (Lazy Evaluation)?
Standard collections store all their items in memory at once. **Iterators** and **Generators** (created with `yield`) produce data lazily (one item at a time only when requested). This allows you to process datasets with millions of records without running out of RAM.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Iterator Protocol**: Covers implementing `__iter__()` and `__next__()` with `StopIteration`.
- **Built-In Functions**: Covers `iter()` and `next()` with fallback defaults.
- **Generator Syntax**: Covers `yield`, `yield from`, `.send()`, `.throw()`, and `.close()`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol, Literal, Final, TypedDict, Callable, TypeVar

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from ../data/raw_transactions.csv


### 🔹 Iterator Protocol: `__iter__()`
An iterable implements `__iter__()` returning an iterator object. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.

**Syntax:** `def __iter__(self): return self`


In [2]:
class SimpleIterable:
    def __init__(self, data): self.data = data
    def __iter__(self):
        return iter(self.data)

print('Iterable created:', list(SimpleIterable([1, 2, 3])))

Iterable created: [1, 2, 3]


### 🔹 Iterator Protocol: `__next__()` & `StopIteration`
An iterator implements `__next__()` returning next items and raising `StopIteration` upon exhaustion. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.

**Syntax:** `def __next__(self): ... raise StopIteration`


In [3]:
class TxStream:
    def __init__(self, records, limit=3):
        self.records = records
        self.limit = limit
        self.idx = 0
    def __iter__(self): return self
    def __next__(self):
        if self.idx >= self.limit:
            raise StopIteration
        r = self.records[self.idx]
        self.idx += 1
        return r['transaction_id']

for tx_id in TxStream(transactions):
    print('Streamed TX via protocol:', tx_id)

Streamed TX via protocol: TX110686
Streamed TX via protocol: TX107170
Streamed TX via protocol: TX108328


### 🔹 Built-in: `iter()`
Retrieves an iterator from an iterable container. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.

**Syntax:** `it = iter(container)`


In [4]:
tx_iter = iter(transactions[:2])
print('Iterator instance:', tx_iter)

Iterator instance: <list_iterator object at 0x000001727F3A6470>


### 🔹 Built-in: `next()` with Default Fallback
Retrieves next value from iterator with optional fallback default avoiding `StopIteration`. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `next(it, default)`


In [5]:
print('next(it):', next(tx_iter)['transaction_id'])
print('next(it):', next(tx_iter)['transaction_id'])
print('next(it, default):', next(tx_iter, 'EXHAUSTED'))

next(it): TX110686
next(it): TX107170
next(it, default): EXHAUSTED


### 🔹 Generator Functions: `yield`
Pauses function execution and yields a value to the caller preserving local frame state. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.

**Syntax:** `def gen(): yield val`


In [6]:
def stream_amounts(records):
    for r in records[:3]:
        yield float(r['transaction_amount'])

for amt in stream_amounts(transactions):
    print('Yielded amount:', amt)

Yielded amount: 1216.33
Yielded amount: 324.99
Yielded amount: 136.66


### 🔹 Sub-Generator Delegation: `yield from`
Delegates iteration to a sub-generator or sub-iterable seamlessly. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.

**Syntax:** `yield from subgen`


In [7]:
def outer_stream(records):
    yield from stream_amounts(records)

print('Yield from collected:', list(outer_stream(transactions)))

Yield from collected: [1216.33, 324.99, 136.66]


### 🔹 Generator Communication: `.send()`
Resumes generator and sends data into the `yield` expression. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.

**Syntax:** `gen.send(val)`


In [8]:
def running_accumulator():
    total = 0.0
    while True:
        val = yield total
        if val is None: break
        total += val

acc = running_accumulator()
next(acc) # Prime generator
print('Sent $100:', acc.send(100.0))
print('Sent $250:', acc.send(250.0))

Sent $100: 100.0
Sent $250: 350.0


### 🔹 Generator Exception Injection: `.throw()`
Raises an exception inside the generator at the point of yield. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Always catch specific exceptions (like `ValueError` or `FileNotFoundError`) rather than a bare `except:` to avoid masking unintended bugs.

**Syntax:** `gen.throw(ValueError)`


In [9]:
def fault_tolerant_gen():
    try:
        yield 1
    except ValueError:
        yield 'RECOVERED_FROM_ERROR'

ft = fault_tolerant_gen()
next(ft)
print('Result after .throw():', ft.throw(ValueError))

Result after .throw(): RECOVERED_FROM_ERROR


### 🔹 Generator Termination: `.close()`
Terminates generator by raising `GeneratorExit` at the point of yield. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.

**Syntax:** `gen.close()`


In [10]:
acc.close()
print('Generator closed successfully.')

Generator closed successfully.


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Memory-Efficient Multi-Stage Generator Pipeline

**Approach:** Build a 3-stage lazy generator pipeline for parsing, filtering, and aggregating transaction records without loading entire lists into RAM.
**Syntax:** `stage3 = (x for x in stage2 if condition)`


In [11]:
def stage1_read(records): yield from records
def stage2_parse(records): yield from ({'id': r['transaction_id'], 'amt': float(r['transaction_amount']), 'fraud': int(r['is_fraud'])} for r in records)
def stage3_filter(records): yield from (r for r in records if r['fraud'] == 1 and r['amt'] > 500.0)

pipeline = stage3_filter(stage2_parse(stage1_read(transactions)))
print('First High-Value Fraud via Pipeline:', next(pipeline))

First High-Value Fraud via Pipeline: {'id': 'TX110960', 'amt': 1805.16, 'fraud': 1}
